In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from itertools import cycle
from sklearn.linear_model import LinearRegression

In [ ]:
resize = 1
plt.rcParams.update({
    "figure.figsize": (6.4*resize, 4.0*resize), # (6.4, 4.8)[4:3] -> (6.4, 4.0)[8:5]
    "font.sans-serif": ["Helvetica", "Nimbus Sans", "Arial", "DejaVu Sans"],
})

In [ ]:
npy_path = "./npy/diff_gpu"

In [ ]:
x_base_5 = np.arange(5, 1001, 5)

base_5 = []
ls_label = ["gpu06", "gpu07", "gpu08", "gpu10", "gpu12", "gpu14"]

for l in ls_label:
    base_5.append(np.load(f"{npy_path}/timePerBatch_imagenet_d18_5_1001_5_{l}.npy"))

In [ ]:
pred_b5 = []

for i in base_5:
    reg_model = LinearRegression().fit(x_base_5.reshape(-1, 1), i)
    pred_b5.append(reg_model.predict(x_base_5.reshape(-1, 1)))
    print("Coefficient:", reg_model.coef_, "Intercept:", reg_model.intercept_)

In [ ]:
plt.figure()

for label, dot, pred in zip(ls_label, base_5, pred_b5):
    plt.scatter(x_base_5, dot, s=5, label=label)#color=next(color_iter))
    plt.plot(x_base_5, pred, "--", label=label)#color=next(color_iter))

plt.xlabel('Batch Size')
plt.ylabel('Time (sec)')
plt.title('Training Time per Batch for ResNet-18 on ImageNet')
plt.legend()

plt.savefig("figure.png", dpi=300, bbox_inches="tight")
plt.show()

### Use Server: [gpu12] / Workers: [gpu06, gpu07, gpu10, gpu14]
- ip: 192.168.33.144

In [ ]:
b_5 = []
ls_coef = []
ls_intercept = []

ls_worker = ["gpu06", "gpu07", "gpu10", "gpu14"]

for l in ls_worker:
    b_5.append(np.load(f"{npy_path}/timePerBatch_imagenet_d18_5_1001_5_{l}.npy"))

for i in b_5:
    reg_model = LinearRegression().fit(x_base_5.reshape(-1, 1), i)
    pred_b5.append(reg_model.predict(x_base_5.reshape(-1, 1)))
    print("Coefficient:", reg_model.coef_, "Intercept:", reg_model.intercept_)
    ls_coef.append(reg_model.coef_[0])
    ls_intercept.append(reg_model.intercept_)

coef = np.mean(ls_coef)
intercept = np.mean(ls_intercept)
print(f"time = {coef:.3f} * batch_size + {intercept:.3f}")